# Boat detection adaptation — free Colab

Fine-tune the existing **YOLOv8** boat model with the prepared dataset. This notebook
uses `best_v2.pt` from the ZIP; it does not download generic YOLO weights.

1. Upload **`boat_adaptation_v1_training.zip`** to the top level of Google Drive (**My Drive**).
2. Select **Runtime → Change runtime type → GPU** (T4 if available).
3. Run cells in order. Accept the Drive-mount prompt.
4. Check the dataset preview, then run the training cell.
5. Download the resulting `best_boat_adapted.pt` using the last cell.

Free Colab GPU availability, hardware and runtime duration vary; finishing in one
session is not guaranteed. Checkpoints are written to Drive each epoch. After a
runtime reset, rerun setup/extraction and training to resume from `last.pt`. Work
since the last completed checkpoint can be lost.

[Google's resource limits](https://research.google.com/colaboratory/faq.html#resource-limits) ·
[Ultralytics checkpoint guidance](https://docs.ultralytics.com/integrations/google-colab/)


## 1. Install the tested Ultralytics version and check the GPU
Keep Colab's supplied PyTorch/CUDA installation. If no GPU is assigned, this cell
stops so you do not accidentally start a slow CPU training run.


In [ ]:
import subprocess
import sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.28"])
import torch
import ultralytics
print("Ultralytics:", ultralytics.__version__, "PyTorch:", torch.__version__)
if not torch.cuda.is_available():
    raise RuntimeError("No GPU assigned. Select Runtime > Change runtime type > GPU, reconnect, and rerun. Free GPU access may be temporarily unavailable.")
print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory GiB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))


## 2. Mount Drive and set paths
If you uploaded the ZIP into a subfolder, change **ZIP_ON_DRIVE** below.
Keep **RUN_NAME** unchanged to resume this experiment. Use a new run name to start
a separate experiment without overwriting previous results.


In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount("/content/drive")

ZIP_ON_DRIVE = Path("/content/drive/MyDrive/boat_adaptation_v1_training.zip")
DRIVE_PROJECT = Path("/content/drive/MyDrive/boat_training/runs")
RUN_NAME = "adaptation_v1"
RUN_DIR = DRIVE_PROJECT / RUN_NAME
DRIVE_EXPORT = Path("/content/drive/MyDrive/boat_training/exports/best_boat_adapted.pt")
DATASET_DIR = Path("/content/datasets/boat_adaptation_v1")
DATA_YAML = DATASET_DIR / "data.yaml"
STARTING_MODEL = DATASET_DIR / "starting_weights/best_v2.pt"
LOCAL_ZIP = Path("/content/boat_adaptation_v1_training.zip")
EXPECTED_ZIP_SHA256 = "f11b752e780c71e3fe921651e30cd90c8fd6ca5037dd5bced1b520f3fc85be79"

if not ZIP_ON_DRIVE.is_file():
    raise FileNotFoundError(f"Upload the prepared ZIP to {ZIP_ON_DRIVE}, or change ZIP_ON_DRIVE to its actual location.")
DRIVE_PROJECT.mkdir(parents=True, exist_ok=True)
print("ZIP:", ZIP_ON_DRIVE)
print("Checkpoints:", RUN_DIR / "weights")


## 3. Copy the ZIP to Colab's local disk, verify it, and extract
Training images are read from local `/content`, avoiding thousands of small reads
from Drive. Only checkpoints/logs are saved directly to Drive. The checksum below
is tied to this delivered dataset ZIP; a deliberately regenerated ZIP needs its
own checksum. After a runtime reset the local dataset must be extracted again.


In [ ]:
import hashlib
import shutil
import zipfile

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def extract_verified_dataset(source_zip, local_zip, destination, expected_sha256):
    source_zip, local_zip, destination = map(Path, (source_zip, local_zip, destination))
    # Reuse only a complete local copy of this exact archive.
    if not local_zip.is_file() or sha256_file(local_zip) != expected_sha256:
        print("Copying ZIP from Drive to local storage...")
        shutil.copyfile(source_zip, local_zip)
    if sha256_file(local_zip) != expected_sha256:
        raise ValueError("ZIP checksum mismatch. Verify the upload or update the checksum only if you intentionally changed the dataset.")
    marker = destination / ".colab_extracted_sha256"
    if not marker.is_file() or marker.read_text().strip() != expected_sha256:
        print("Extracting verified archive...")
        destination.parent.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(local_zip) as archive:
            base = destination.parent.resolve()
            for member in archive.infolist():
                if not (base / member.filename).resolve().is_relative_to(base):
                    raise ValueError("Archive contains an unsafe path")
            archive.extractall(base)
        marker.write_text(expected_sha256 + "\n")
    expected_counts = {"train": 3178, "val": 384, "test": 384}
    for split, expected in expected_counts.items():
        images = [p for p in (destination / split / "images").iterdir() if p.suffix.lower() in (".jpg", ".jpeg", ".png")]
        assert len(images) == expected, (split, len(images), expected)
        assert all((destination / split / "labels" / (p.stem + ".txt")).is_file() for p in images)
        print(split, len(images), "images; corresponding labels present")
    assert (destination / "data.yaml").is_file()
    assert (destination / "starting_weights/best_v2.pt").is_file()
    return destination

extract_verified_dataset(ZIP_ON_DRIVE, LOCAL_ZIP, DATASET_DIR, EXPECTED_ZIP_SHA256)


## 4. Verify class mapping and inspect the labels
Classes must stay `0=leader`, `1=follower`. The mask-based boxes describe visible
boat meshes through transparent spray; wake/spray are excluded from boat boxes.
Some visible parts of the camera's own vessel are also labeled. Inspect the
preview before training. Entire simulation runs are separated across splits.


In [ ]:
from ultralytics import YOLO
from ultralytics.data.utils import check_det_dataset
from IPython.display import display
from PIL import Image

data = check_det_dataset(str(DATA_YAML), autodownload=False)
assert data["names"] == {0: "leader", 1: "follower"}, data["names"]
starting_model = YOLO(str(STARTING_MODEL))
assert starting_model.names == data["names"], starting_model.names
print("Classes:", data["names"])
print("Starting architecture:", starting_model.model.yaml.get("yaml_file", "YOLOv8 checkpoint"))
display(Image.open(DATASET_DIR / "train-preview.jpg"))


## 5. Fine-tune or resume
A conservative free-GPU starting point is **batch 8**, **image size 640**, and
**50 maximum epochs with early stopping**. If GPU memory is exhausted, restart
the runtime, rerun setup, and set BATCH to 4. Keep the same run name to resume.
Do not start several training cells concurrently.

`last.pt` and `best.pt` are written to Drive after completed epochs; additional
periodic checkpoints are kept every five epochs. Resume restores the original
training schedule and optimizer. Changing EPOCHS does not extend a resumed run;
use a new run name for a fresh experiment. A completed/stripped `last.pt` is not
resumed automatically. If your session is cut off mid-write, choose an earlier
periodic checkpoint by setting CHECKPOINT_TO_RESUME to its full Drive path.

[Ultralytics resume documentation](https://docs.ultralytics.com/modes/train/#resuming-interrupted-trainings)


In [ ]:
EPOCHS = 50
BATCH = 8
IMGSZ = 640
CHECKPOINT_TO_RESUME = RUN_DIR / "weights/last.pt"
BEST = RUN_DIR / "weights/best.pt"

if CHECKPOINT_TO_RESUME.is_file():
    model = YOLO(str(CHECKPOINT_TO_RESUME))
    checkpoint = model.ckpt
    resumable = checkpoint.get("epoch", -1) >= 0 and checkpoint.get("optimizer") is not None
    if resumable:
        print("Resuming:", CHECKPOINT_TO_RESUME)
        results = model.train(resume=True, data=str(DATA_YAML), batch=BATCH,
                              device=0, workers=2, cache=False, save_period=5)
    elif BEST.is_file():
        print("This run has a completed/non-resumable checkpoint. Training was not restarted.")
        print("Use the next cells to evaluate/export best.pt, or choose a new RUN_NAME for a fresh experiment.")
    else:
        raise RuntimeError("Checkpoint is not resumable and best.pt is missing. Select an earlier periodic checkpoint or use a new RUN_NAME.")
else:
    model = YOLO(str(STARTING_MODEL))
    print("Fine-tuning original boat weights; outputs:", RUN_DIR)
    results = model.train(
        data=str(DATA_YAML), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
        device=0, workers=2, cache=False, amp=True,
        optimizer="AdamW", lr0=0.001, patience=12, seed=628100,
        mosaic=0.3, close_mosaic=8, hsv_h=0.015, hsv_s=0.35, hsv_v=0.3,
        scale=0.3, translate=0.08, flipud=0.0, fliplr=0.5,
        project=str(DRIVE_PROJECT), name=RUN_NAME, exist_ok=True,
        save=True, save_period=5,
    )
print("Best checkpoint:", BEST)
print("Last checkpoint:", RUN_DIR / "weights/last.pt")


## 6. Optional final test
Leave RUN_FINAL_TEST false while selecting settings. After choosing the final
checkpoint, set it true and run once. Validation/test use held-out full camera
frames; all contain boats, so whole empty-ocean episodes still need a separate
operational check. Detection metrics alone do not prove that formation control
or leader identity switching has been fixed.


In [ ]:
RUN_FINAL_TEST = False
if RUN_FINAL_TEST:
    if not BEST.is_file():
        raise FileNotFoundError("No adapted best.pt yet. Finish or resume training first.")
    metrics = YOLO(str(BEST)).val(
        data=str(DATA_YAML), split="test", imgsz=IMGSZ, batch=BATCH,
        device=0, workers=2, project=str(DRIVE_PROJECT),
        name=RUN_NAME + "_final_test", exist_ok=True,
    )
    print(metrics.results_dict)
else:
    print("Final test skipped. Set RUN_FINAL_TEST=True only when ready.")


## 7. Save a clearly named model and download it
The exported file is separate from the original model. Copy it back to the Unity
machine after training and compare it with the old checkpoint before deployment.
Your Drive run directory also contains the logs, plots and resumable checkpoints.


In [ ]:
from google.colab import files
if not BEST.is_file():
    raise FileNotFoundError("No adapted best.pt available. Finish or resume the training cell first.")
DRIVE_EXPORT.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(BEST, DRIVE_EXPORT)
print("Saved:", DRIVE_EXPORT)
files.download(str(DRIVE_EXPORT))
